In [ ]:
from fastmcp import FastMCP
import gspread
import json
import uuid
from datetime import datetime

mcp = FastMCP("SheetsMCP")

# Google Sheets setup
gc = gspread.service_account(
    filename="credentials.json"
)

sheet = gc.open(
    "RestaurantOrders"
).sheet1
print("Google Sheets connected successfully!",sheet.title)

@mcp.tool()
def log_order(
    user_id: str,
    items: list,
    total: float
):
    """
    Save complete order to Google Sheet
    """

    order_id = str(uuid.uuid4())[:8]

    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    sheet.append_row([
        order_id,
        user_id,
        json.dumps(items),
        total,
        timestamp
    ])

    return {
        "order_id": order_id,
        "message": "Order saved successfully"
    }


@mcp.tool()
def get_order(order_id: str):
    """
    Fetch a specific order
    """

    rows = sheet.get_all_records()

    for row in rows:

        if str(row.get("order_id")) == str(order_id):
            return row

    return {
        "error": "Order not found"
    }


@mcp.tool()
def get_orders():
    """
    Fetch all orders
    """

    return sheet.get_all_records()


@mcp.tool()
def get_user_orders(user_id: str):
    """
    Fetch all orders of a user
    """

    rows = sheet.get_all_records()

    return [
        row
        for row in rows
        if str(row.get("user_id")) == str(user_id)
    ]


